# IWI experiment · Download Landsat, nighttime lights and MODIS NDVI for each DHS cluster

For every DHS cluster interviewed in 2000 or later, this notebook creates `Dataset/<cluster_id>/` and saves three GeoTIFFs from Earth Engine in it. All three use the same 224 × 224 grid: EPSG:4326, about 30 m pixels, a 6.72 km box from `buffer(112*30).bounds()` around the cluster.

| file | content | bands |
|---|---|---|
| `Landsat.tif` | annual median of cloud-masked Landsat C2 L2, raw DN (SR = DN × 0.0000275 − 0.2, TEMP1 = DN × 0.00341802 + 149 K) | BLUE, GREEN, RED, NIR, SWIR1, SWIR2, TEMP1 |
| `NL.tif` | annual median nighttime lights: DMSP-OLS `avg_vis` up to 2013, VIIRS VCMSLCFG `avg_rad` from 2014 | NL |
| `MODIS_NDVI.tif` | the 23 MOD13Q1.061 16-day NDVI composites of the survey year, × 0.0001 | NDVI_01 … NDVI_23 |

Pixels without data are `-999999`, which is also the GeoTIFF nodata value. Every file stores its band names and the cluster id, year and source as tags.

**Landsat gap filling.** Level 1 is the composite of L5 + L7 (SLC-on) + L8 + L9 in the survey year. From June 2003 to early 2013 that leaves only Landsat 5, which covers parts of Africa sparsely, and 2012 has almost no scenes (Landsat 5 stopped imaging in November 2011, Landsat 8 data start in 2013). Pixels that level 1 leaves empty are filled from level 2 (adds L7 SLC-off) and then level 3 (level 2 over year − 1 … year + 1). The filled version is requested only when level 1 has holes. `download_log.csv` records the share of pixels from each level.

**Running.** Check `CSV_PATH` in Settings, then run the cells in order. Section 6 tests one cluster; section 7 downloads the rest with `Pool(N_WORKERS)` and can be re-run at any time, because it only fetches files that are still missing.

# Import libraries

In [ ]:
import os
import csv
import datetime
import multiprocessing
from collections import Counter

# The GPU image sets PROJ_LIB=/usr/share/proj, which has no proj.db, and rasterio then cannot
# read or write a CRS. Drop such a setting so rasterio uses the PROJ data it ships with.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import ee
import requests
import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile
from affine import Affine
from retry import retry
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Settings

In [ ]:
# ---- input / output ----------------------------------------------------------------
CSV_PATH = '/path/to/data/dhs_data_cluster.csv'  # <-- check
OUT_DIR  = '/path/to/data/Dataset/'
LOG_CSV  = '/path/to/data/download_log.csv'

MIN_YEAR = 2000                      # keep clusters interviewed in this year or later

FILE_LANDSAT = 'Landsat.tif'         # the three files in every cluster folder
FILE_NL      = 'NL.tif'
FILE_MODIS   = 'MODIS_NDVI.tif'

# ---- patch grid, the same for all three files ----------------------------------------
PATCH_PX = 224                       # 224 x 224 pixels
BUFFER_M = 112 * 30                  # ROI = point.buffer(112*30).bounds(): 6.72 km box, ~30 m pixels
NODATA   = -999999.0                 # value of pixels without data (also the GeoTIFF nodata value)

# ---- Landsat ---------------------------------------------------------------------------
LANDSAT_FILL_GAPS = True             # False = level 1 only

# ---- Earth Engine and parallel download ------------------------------------------------
EE_PROJECT = 'your-earth-engine-project'   # Google Cloud project registered for Earth Engine
EE_URL     = 'https://earthengine-highvolume.googleapis.com'
N_WORKERS  = 32

# Earth Engine authentication

In [ ]:
# Trigger the authentication flow (returns at once if a valid token is already saved).
ee.Authenticate()

# Initialize the library on the high-volume endpoint.
ee.Initialize(project=EE_PROJECT, opt_url=EE_URL)

# 1. DHS clusters from 2000 onwards

In [ ]:
TableDHS = pd.read_csv(CSV_PATH)
# the older CSV version used ClusterID / year.of.interview / month.of.interview
TableDHS = TableDHS.rename(columns={'ClusterID': 'cluster_id', 'year.of.interview': 'year',
                                    'month.of.interview': 'month'})
print(TableDHS.columns.tolist())
print('rows in CSV             :', len(TableDHS))

TableDHS = TableDHS[TableDHS['year'] >= MIN_YEAR]
print(f'year >= {MIN_YEAR}          :', len(TableDHS))

# clusters that cannot be located: missing coordinates or the DHS (0, 0) placeholder
no_xy = TableDHS[['lon', 'lat']].isna().any(axis=1) | ((TableDHS['lon'] == 0) & (TableDHS['lat'] == 0))
TableDHS = TableDHS[~no_xy]
dup = TableDHS['cluster_id'].duplicated(keep='first')
TableDHS = TableDHS[~dup].reset_index(drop=True)
TableDHS['year'] = TableDHS['year'].astype(int)
print('dropped, no coordinates :', int(no_xy.sum()))
print('dropped, duplicate id   :', int(dup.sum()))
print('clusters                :', len(TableDHS))
print(TableDHS.groupby('year').size().to_string())
TableDHS.head()

# 2. One folder per cluster

In [ ]:
def cluster_dir(cluster_id):
    # the cluster_id is the folder name; only '/' would break the path
    return os.path.join(OUT_DIR, str(cluster_id).replace('/', '_'))

def cluster_files(cluster_id):
    folder = cluster_dir(cluster_id)
    return {'landsat': os.path.join(folder, FILE_LANDSAT),
            'modis':   os.path.join(folder, FILE_MODIS),
            'nl':      os.path.join(folder, FILE_NL)}

os.makedirs(OUT_DIR, exist_ok=True)
for cid in TableDHS['cluster_id']:
    os.makedirs(cluster_dir(cid), exist_ok=True)
print('cluster folders:', sum(os.path.isdir(cluster_dir(c)) for c in TableDHS['cluster_id']), 'in', OUT_DIR)

# 3.–5. Earth Engine collections and images

Landsat and nighttime lights follow `Application_Download_LandsatNL_CustomROI` / `40_Download_Landsat_Patches` and `08_Download_NL_DMSP_VIIRS`. MODIS follows `01_Download_MODIS`, but builds the 23 bands from the composite dates, so a year with a missing composite still gets 23 bands. Year 2000 is such a year: MODIS starts on 18 February 2000, so NDVI_01–NDVI_03 are empty.

In [ ]:
# ---- Landsat Collection 2 Level 2
CollectionL5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
CollectionL7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterDate('1999-01-01', '2003-05-31')          # SLC-on
CollectionL7_SLCoff = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterDate('2003-05-31', '2100-01-01')  # gap filling only
CollectionL8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
CollectionL9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

LANDSAT_BANDS = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'TEMP1']
BANDS_L57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ST_B6']    # Landsat 5 TM, 7 ETM+
BANDS_L89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']   # Landsat 8, 9

def mask_L8_SR(img):
    # QA_PIXEL bit 3 = cloud, bit 4 = cloud shadow (same bits for Landsat 5, 7, 8 and 9 in Collection 2)
    qa = img.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3)
    shadow = qa.bitwiseAnd(1 << 4)
    Mask = (cloud.neq(0)).Or(shadow.neq(0)).uint8()
    return img.updateMask(Mask.eq(0))

# ---- MODIS Terra vegetation indices, 16 days, 250 m: 23 composites per year (DOY 1, 17, ..., 353)
CollectionModisTerra = ee.ImageCollection("MODIS/061/MOD13Q1")
MODIS_BANDS = ['NDVI_%02d' % k for k in range(1, 24)]
MODIS_EMPTY = ee.Image.constant(0).toInt16().rename('NDVI').updateMask(0)   # stands in for a missing composite

# ---- Nighttime lights
CollectionVIIRS = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG').select(['avg_rad'], ['NL'])  # 2014-01 onwards
CollectionDMSP = ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS").select(['avg_vis'], ['NL'])       # 1992-2013
CollectionNL = CollectionVIIRS.merge(CollectionDMSP)

LANDSAT_LEVELS = (1, 2, 3) if LANDSAT_FILL_GAPS else (1,)

In [ ]:
def year_range(first_year, last_year):
    """Date strings covering 1 January of first_year to 31 December of last_year."""
    return f'{first_year}-01-01', f'{last_year + 1}-01-01'


def landsat_collection(year, roi, level):
    """Cloud-masked Landsat scenes over roi, renamed to LANDSAT_BANDS.
    level 1: L5 + L7 SLC-on + L8 + L9 in the survey year (40_Download_Landsat_Patches)
    level 2: level 1 + L7 SLC-off
    level 3: level 2 over year-1 .. year+1"""
    start, stop = year_range(year - 1, year + 1) if level == 3 else year_range(year, year)

    def prep(collection, bands):
        return (collection.filterDate(start, stop).filterBounds(roi)
                .map(lambda x: mask_L8_SR(x).select(bands, LANDSAT_BANDS)))

    col = (prep(CollectionL9, BANDS_L89).merge(prep(CollectionL8, BANDS_L89))
           .merge(prep(CollectionL7, BANDS_L57)).merge(prep(CollectionL5, BANDS_L57)))
    if level >= 2:
        col = col.merge(prep(CollectionL7_SLCoff, BANDS_L57))
    return col


def landsat_image(year, roi, levels):
    """Median composite of the first level; pixels it leaves empty are filled from the next levels.
    The extra band LEVEL says which level each pixel came from (0 = none). It goes to the log, not the file."""
    composites = [landsat_collection(year, roi, lvl).median() for lvl in levels]
    img = composites[0]
    for comp in composites[1:]:
        img = img.unmask(comp)
    level = ee.Image.constant(0)
    for lvl, comp in reversed(list(zip(levels, composites))):
        level = level.where(comp.select('BLUE').mask(), lvl)
    return img.addBands(level.rename('LEVEL'))


def modis_ndvi(year):
    """NDVI_01..NDVI_23 = the MOD13Q1 composites starting on DOY 1, 17, ..., 353 of the year, x 0.0001.
    A composite that does not exist becomes an empty band."""
    col = CollectionModisTerra.filterDate(*year_range(year, year)).select('NDVI')
    jan1 = datetime.date(year, 1, 1)
    bands = []
    for k, name in enumerate(MODIS_BANDS):
        start = jan1 + datetime.timedelta(days=16 * k)
        stop = start + datetime.timedelta(days=16)
        one = col.filterDate(start.isoformat(), stop.isoformat())
        bands.append(ee.ImageCollection([MODIS_EMPTY]).merge(one).mosaic().rename(name))
    return ee.Image.cat(*bands).multiply(0.0001)


def nl_image(year, roi):
    """Annual median nighttime lights: DMSP-OLS up to 2013, VIIRS from 2014 (08_Download_NL_DMSP_VIIRS)."""
    return CollectionNL.filterDate(*year_range(year, year)).filterBounds(roi).median()


@retry(tries=6, delay=2, backoff=2, max_delay=60, jitter=(0, 2), logger=None)
def count_images(year, roi):
    """How many images each product has for this cluster (one Earth Engine request)."""
    n = {'n_modis': CollectionModisTerra.filterDate(*year_range(year, year)).size(),
         'n_viirs': CollectionVIIRS.filterDate(*year_range(year, year)).size(),
         'n_dmsp':  CollectionDMSP.filterDate(*year_range(year, year)).size()}
    for lvl in LANDSAT_LEVELS:
        n[f'n_landsat_{lvl}'] = landsat_collection(year, roi, lvl).size()
    return ee.Dictionary(n).getInfo()

# 6. Download one cluster: three files per folder

In [ ]:
DOWNLOAD_PARAMS = {'crs': 'EPSG:4326', 'dimensions': f'{PATCH_PX}x{PATCH_PX}', 'format': 'GEO_TIFF'}


@retry(tries=6, delay=2, backoff=2, max_delay=60, jitter=(0, 2), logger=None)
def download(image, roi):
    """Download image over roi as a float32 array (bands, PATCH_PX, PATCH_PX), north up, NODATA where masked."""
    url = image.unmask(NODATA).float().getDownloadURL(dict(DOWNLOAD_PARAMS, region=roi))
    r = requests.get(url, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f'HTTP {r.status_code}: {r.text[:300]}')
    with MemoryFile(r.content) as mem, mem.open() as src:
        data, crs, transform = src.read(), src.crs, src.transform
    if data.shape[1:] != (PATCH_PX, PATCH_PX) or crs is None:
        raise RuntimeError(f'unexpected download: shape {data.shape}, crs {crs}')
    if transform.e > 0:                                   # south-up grid: flip to north-up
        data = data[:, ::-1, :]
        transform = transform * Affine.translation(0, PATCH_PX) * Affine.scale(1, -1)
    data = np.where(np.isfinite(data), data, NODATA).astype('float32')
    return data, crs, transform


def write_tif(path, data, crs, transform, band_names, tags):
    """Write a compressed GeoTIFF with band names and tags. It is written to path + '.part' and
    renamed at the end, so an interrupted run never leaves a half-written file under the final name."""
    tmp = path + '.part'
    with rasterio.open(tmp, 'w', driver='GTiff', width=PATCH_PX, height=PATCH_PX, count=len(band_names),
                       dtype='float32', crs=crs, transform=transform, nodata=NODATA,
                       compress='deflate', predictor=3, interleave='band') as dst:
        dst.write(data)
        for i, name in enumerate(band_names, start=1):
            dst.set_band_description(i, name)
        dst.update_tags(**{k: str(v) for k, v in tags.items()})
    os.replace(tmp, path)


# Each save_* function writes one file and returns False when the product has no image for that year.

def save_landsat(path, year, ROI, n, tags, rec):              # 3. Landsat, 7 bands
    levels, prev = [], 0                # levels that add scenes (they are nested, so counts only grow)
    for lvl in LANDSAT_LEVELS:
        if n[f'n_landsat_{lvl}'] > prev:
            levels.append(lvl)
            prev = n[f'n_landsat_{lvl}']
    if not levels:
        return False
    data, crs, tr = download(landsat_image(year, ROI, levels[:1]), ROI)
    if len(levels) > 1 and (data[:-1] == NODATA).any():         # holes: fill them from the next levels
        data, crs, tr = download(landsat_image(year, ROI, levels), ROI)
    level, data = data[-1], data[:-1]
    rec['landsat_levels'] = ' '.join(str(lv) for lv in levels)
    for lv in (1, 2, 3):
        rec[f'landsat_frac_{lv}'] = round(float((level == lv).mean()), 4)
    rec['landsat_gap'] = round(float((data == NODATA).any(axis=0).mean()), 4)
    write_tif(path, data, crs, tr, LANDSAT_BANDS, dict(
        tags, source='Landsat C2 L2 annual median (raw DN), cloud and shadow masked',
        scale='SR = DN*0.0000275-0.2; TEMP1 = DN*0.00341802+149 (K)',
        levels=rec['landsat_levels'],
        fraction_per_level=' '.join(str(rec[f'landsat_frac_{lv}']) for lv in (1, 2, 3)),
        scenes_per_level=' '.join(str(n[f'n_landsat_{lv}']) for lv in LANDSAT_LEVELS)))
    return True


def save_modis(path, year, ROI, n, tags, rec):                # 4. MODIS NDVI, 23 bands
    if n['n_modis'] == 0:
        return False
    data, crs, tr = download(modis_ndvi(year), ROI)
    write_tif(path, data, crs, tr, MODIS_BANDS, dict(
        tags, source='MODIS/061/MOD13Q1 NDVI x 0.0001; NDVI_k = composite starting on DOY 16(k-1)+1',
        n_composites=n['n_modis']))
    return True


def save_nl(path, year, ROI, n, tags, rec):                   # 5. Nighttime lights, 1 band
    if n['n_viirs'] + n['n_dmsp'] == 0:
        return False
    sensor = 'VIIRS' if n['n_viirs'] > 0 else 'DMSP'
    data, crs, tr = download(nl_image(year, ROI), ROI)
    write_tif(path, data, crs, tr, ['NL'], dict(
        tags, sensor=sensor, n_images=n['n_viirs'] + n['n_dmsp'],
        source=('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG avg_rad, annual median' if sensor == 'VIIRS'
                else 'NOAA/DMSP-OLS/NIGHTTIME_LIGHTS avg_vis, annual median')))
    return True


SAVE = {'landsat': save_landsat, 'modis': save_modis, 'nl': save_nl}


def getResult(arg, point):
    """Download the files a cluster is still missing. Returns one row of download_log.csv."""
    cid, year = str(point['cluster_id']), int(point['year'])
    lon, lat = float(point['lon']), float(point['lat'])
    paths = cluster_files(cid)
    rec = {'cluster_id': cid, 'year': year, 'lon': lon, 'lat': lat}
    todo = [k for k, p in paths.items() if not os.path.exists(p)]
    if not todo:
        rec['status'] = 'exists'
        return rec

    ROI = ee.Geometry.Point([lon, lat]).buffer(BUFFER_M).bounds()
    try:
        os.makedirs(cluster_dir(cid), exist_ok=True)
        n = count_images(year, ROI)
    except Exception as e:
        rec['status'], rec['error'] = 'error', f'{type(e).__name__}: {e}'[:500]
        return rec
    rec.update(n)

    tags = {'cluster_id': cid, 'year': year, 'lon': lon, 'lat': lat}
    missing, errors = [], []
    for product in todo:                # one product failing does not stop the others
        try:
            if not SAVE[product](paths[product], year, ROI, n, tags, rec):
                missing.append(product)
        except Exception as e:
            errors.append(f'{product}: {type(e).__name__}: {e}'[:300])
    if errors:
        rec['status'] = 'error'
        rec['error'] = ' | '.join(errors + [f'no data: {m}' for m in missing])
    else:
        rec['status'] = 'ok' if not missing else 'no data: ' + ', '.join(missing)
    return rec


LOG_FIELDS = ['time', 'cluster_id', 'year', 'lon', 'lat', 'status', 'landsat_levels',
              'landsat_frac_1', 'landsat_frac_2', 'landsat_frac_3', 'landsat_gap',
              'n_landsat_1', 'n_landsat_2', 'n_landsat_3', 'n_modis', 'n_viirs', 'n_dmsp', 'error']

def append_log(rec):
    """Append one getResult row to LOG_CSV (writes the header the first time)."""
    if rec['status'] == 'exists':       # nothing was downloaded
        return
    new = not os.path.exists(LOG_CSV)
    with open(LOG_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS, extrasaction='ignore')
        if new:
            w.writeheader()
        w.writerow(dict(rec, time=datetime.datetime.now().isoformat(timespec='seconds')))

In [ ]:
rec = getResult(100, TableDHS.iloc[100])
append_log(rec)
rec

In [ ]:
def show_cluster(cluster_id):
    """Print the shape, band names and tags of a cluster's files and plot them."""
    files = cluster_files(cluster_id)
    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(cluster_id)
    if os.path.exists(files['landsat']):
        with rasterio.open(files['landsat']) as src:
            ls = src.read(masked=True)
            print(FILE_LANDSAT, ls.shape, src.descriptions, src.tags(), '\n')
        rgb = np.clip((ls[[2, 1, 0]].filled(0) * 0.0000275 - 0.2) / 0.3, 0, 1)   # DN -> reflectance 0..0.3
        ax[0].imshow(np.moveaxis(rgb, 0, -1))
        ax[0].set_title('Landsat RGB')
    if os.path.exists(files['nl']):
        with rasterio.open(files['nl']) as src:
            nl = src.read(1, masked=True)
            print(FILE_NL, nl.shape, src.tags(), '\n')
        fig.colorbar(ax[1].imshow(nl, cmap='magma'), ax=ax[1])
        ax[1].set_title('NL')
    if os.path.exists(files['modis']):
        with rasterio.open(files['modis']) as src:
            nd = src.read(masked=True)
            print(FILE_MODIS, nd.shape, src.tags())
        fig.colorbar(ax[2].imshow(nd.mean(axis=0), cmap='YlGn', vmin=0, vmax=1), ax=ax[2])
        ax[2].set_title('mean NDVI')
        ax[3].plot(range(1, 24), nd[:, PATCH_PX // 2, PATCH_PX // 2], 'o-')
        ax[3].set_xlabel('16-day composite')
        ax[3].set_title('NDVI at the cluster')
    plt.show()

show_cluster(TableDHS.iloc[100]['cluster_id'])

# 7. Parallel download

Each worker runs `getResult` for one cluster at a time and the results go to `download_log.csv` (one row per attempt). Stop the cell at any time; running it again skips the clusters that already have all three files. Clusters with `no data` or `error` are tried again on the next run.

In [ ]:
def _init_worker():
    # each worker opens its own connections instead of sharing the parent's forked ones
    try:
        ee.Initialize(project=EE_PROJECT, opt_url=EE_URL)
    except Exception:
        pass                            # keep the session inherited from this notebook

def _run(task):
    return getResult(*task)

tasks = [(i, row) for i, row in TableDHS.iterrows()
         if not all(os.path.exists(p) for p in cluster_files(row['cluster_id']).values())]
print(f'{len(TableDHS) - len(tasks)} clusters complete, {len(tasks)} to download')

status = Counter()
with multiprocessing.get_context('fork').Pool(N_WORKERS, initializer=_init_worker) as pool:
    bar = tqdm(pool.imap_unordered(_run, tasks), total=len(tasks), smoothing=0.05)
    for rec in bar:
        append_log(rec)
        status[rec['status'].split(':')[0]] += 1
        bar.set_postfix(status, refresh=False)
print(dict(status))

# 8. Check the result

In [ ]:
complete = [all(os.path.exists(p) for p in cluster_files(c).values()) for c in TableDHS['cluster_id']]
print(f'{sum(complete)} of {len(TableDHS)} cluster folders have all three files\n')

log = pd.read_csv(LOG_CSV).drop_duplicates('cluster_id', keep='last')
print(log['status'].value_counts().to_string(), '\n')

# share of Landsat pixels per level and still empty, by survey year
print(log.groupby('year')[['landsat_frac_1', 'landsat_frac_2', 'landsat_frac_3', 'landsat_gap']].mean().round(3).to_string())
log[log['status'] != 'ok'].head(20)